In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
let divide (numerator: float) (denominator: float) =
    if denominator = 0.0 then failwith "Division by zero"
    numerator / denominator

# An EBNF based Parser for Arithmetic Expressions

In this notebook we implement an <span style="font-variant:small-caps;">Ebnf</span> recursive-descend parser for arithmetic expressions.  This parser implements the following <span style="font-variant:small-caps;">Ebnf</span> grammar:
$$
  \begin{eqnarray*}
  \mathrm{expr}    & \rightarrow & \mathrm{product}\;\;\bigl((\texttt{'+'}\;|\;\texttt{'-'})\;\; \mathrm{product}\bigr)^* \\[0.2cm]
  \mathrm{product} & \rightarrow & \mathrm{factor} \;\;\bigl((\texttt{'*'}\;|\;\texttt{'/'})\;\; \mathrm{factor}\bigr)^*  \\[0.2cm]   
  \mathrm{factor}  & \rightarrow & \texttt{'('} \;\;\mathrm{expr} \;\;\texttt{')'}                             \\
                   & \mid        & \texttt{NUMBER} 
  \end{eqnarray*}
$$

## Implementing a Scanner

The .NET `Regex` class provides regular expressions. Each match is anchored to prevent silently skipping invalid input.

In [ ]:
open System
open System.Globalization
open System.Text.RegularExpressions

// \G anchors every match at the previous match: illegal characters cannot disappear.
let tokenize (text: string) =
    let pattern = Regex(@"\G(?:\s+|[1-9][0-9]*|0|[a-z][0-9]*|[-+*/()])")
    let rec scan position tokens =
        if position = text.Length then List.rev tokens
        else
            let token = pattern.Match(text, position)
            if not token.Success then failwithf "Illegal character at column %d" (position + 1)
            let rest = position + token.Length
            if String.IsNullOrWhiteSpace token.Value then scan rest tokens
            else scan rest (token.Value :: tokens)
    scan 0 []

`tokenize` returns the token list. Whitespace is discarded; an unknown character raises a useful error.

In [ ]:
printfn "%A" (tokenize "12 * 13 + 14 * 4 / 6 - 7")

## Implementing the Recursive Descend Parser

F# defines mutually recursive functions with `let rec` and `and`. Their types are inferred without forward declarations. The grammar rules below are followed by the whole mutually recursive group.

The function `parse` takes a string `s` as input and parses this string according to the recursive grammar
shown above.

The function `parseExpr(TL)` takes a list of tokens `TL` and tries to parse an expresssion according to the following
<span style="font-variant:small-caps;">Ebnf</span> grammar rule:
$$ \mathrm{expr} \;\rightarrow\; \mathrm{product}\;\;\bigl((\texttt{'+'}\;|\;\texttt{'-'})\;\; \mathrm{product}\bigr)^* $$
It returns the value of the expression and a list of all the tokens that have not been consumed during parsing.

The function `parseProduct(TL)` takes a list of tokens `TL` and tries to parse a product according to the following
<span style="font-variant:small-caps;">Ebnf</span> grammar rule:
$$ \mathrm{product} \;\rightarrow\; \mathrm{factor} \;\;\bigl((\texttt{'*'}\;|\;\texttt{'/'})\;\; \mathrm{factor}\bigr)^*  $$
It returns the value of the product and a list of all the tokens that have not been consumed during parsing.

The function `parseFactor` implements the following grammar rules:
$$
  \begin{eqnarray*}
  \mathrm{factor}      & \;\rightarrow\; & \texttt{'('} \;\;\mathrm{expr} \;\;\texttt{')'}                \\
                       & \;\mid          & \;\texttt{NUMBER} 
  \end{eqnarray*}
$$

It takes one argument:
- `TL` is the list of tokens that still need to be consumed.

It returns a pair of the form `(value, Rest)` where
- `value` is the result of evaluating the arithmetical expression
  that is represented by `TL` and
- `Rest` is a list of those tokens that have not been consumed while trying to parse a factor.

In [ ]:
let rec parseExpr tokens =
    let first, rest = parseProduct tokens
    let rec repeat value remaining =
        match remaining with
        | ("+" | "-" as op) :: tail ->
            let next, rest = parseProduct tail
            repeat (if op = "+" then value + next else value - next) rest
        | _ -> value, remaining
    repeat first rest
and parseProduct tokens =
    let first, rest = parseFactor tokens
    let rec repeat value remaining =
        match remaining with
        | ("*" | "/" as op) :: tail ->
            let next, rest = parseFactor tail
            repeat (if op = "*" then value * next else divide value next) rest
        | _ -> value, remaining
    repeat first rest
and parseFactor tokens =
    match tokens with
    | "(" :: tail ->
        let value, rest = parseExpr tail
        match rest with
        | ")" :: remaining -> value, remaining
        | _ -> failwith "Parse error: expected ')'"
    | number :: tail ->
        match Double.TryParse(number, CultureInfo.InvariantCulture) with
        | true, value -> value, tail
        | _ -> failwithf "Parse error: expected a number, got %s" number
    | [] -> failwith "Parse error: expected a factor"

let parse text =
    let value, rest = parseExpr (tokenize text)
    if not rest.IsEmpty then failwithf "Unconsumed tokens: %A" rest
    value

## Testing

In [ ]:
let test text expected =
    let actual = parse text
    checkNotebook (abs(actual - expected) < 1e-9)
    actual

In [ ]:
test "11+22*(33-44)/(5-10*5/(4-3))" (11.0 + 22.0 * (33.0 - 44.0) / (5.0 - 10.0 * 5.0 / (4.0 - 3.0))) |> printfn "%g"

In [ ]:
test "0*11+22*(33-44)/(5-10*5/(4-3))" (22.0 * (33.0 - 44.0) / (5.0 - 10.0 * 5.0 / (4.0 - 3.0))) |> printfn "%g"

In [ ]:
let rejects text =
    try parse text |> ignore; false
    with _ -> true
checkNotebook (rejects "1 + @ 2")
checkNotebook (rejects "1 +")
checkNotebook (rejects "(1 + 2")
checkNotebook (rejects "1 2")